[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgcalvo/SP1306/blob/main/edps-elipticas/MDF_converg.ipynb)

# Convergencia del método de diferencias finitas

Traducción de `MDF_v1_converg.m`.

Mismo problema y mismo ensamblaje que [`MDF_test.ipynb`](MDF_test.ipynb), pero
resuelto sobre una sucesión de mallas cada vez más finas, $M = 4, 8, \dots, 128$,
midiendo el error contra la solución exacta.

La teoría dice que la fórmula de 5 puntos es de orden 2, es decir
$\text{error} \sim C h^2$. En escala log-log eso es una recta de pendiente 2, y
cada vez que $h$ se parte a la mitad el error debería dividirse entre 4.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve

## Entradas

In [ ]:
uex = lambda x, y: np.sin(np.pi*x)*np.sin(np.pi*y)             # sol. exacta
f   = lambda x, y: 2*np.pi**2*np.sin(np.pi*x)*np.sin(np.pi*y)  # lado derecho

## El ensamblaje, como función

Es el mismo cuerpo del ciclo del otro notebook, envuelto en una función para
poder llamarlo con distintos valores de `M`.

In [ ]:
def resolver(M):
    """Arma y resuelve el sistema para una malla de M intervalos.

    Devuelve (h, error en norma infinito)."""
    h    = 1/M
    dimA = (M-1)**2

    ii, jj, ss = [], [], []
    b      = np.zeros(dimA)
    UexMat = np.zeros((M-1, M-1))

    for j in range(1, M):
        for i in range(1, M):
            pos = (j-1)*(M-1) + (i-1)
            b[pos] = h**2 * f(i*h, j*h)
            UexMat[i-1, j-1] = uex(i*h, j*h)

            ii.append(pos); jj.append(pos); ss.append(4.0)
            if i != 1:
                ii.append(pos); jj.append(pos-1); ss.append(-1.0)
            if i != M-1:
                ii.append(pos); jj.append(pos+1); ss.append(-1.0)
            if j != 1:
                ii.append(pos); jj.append(pos-(M-1)); ss.append(-1.0)
            if j != M-1:
                ii.append(pos); jj.append(pos+(M-1)); ss.append(-1.0)

    A  = coo_matrix((ss, (ii, jj)), shape=(dimA, dimA)).tocsr()
    uh = spsolve(A, b)
    return h, np.linalg.norm(uh - UexMat.ravel(order='F'), np.inf)

## El estudio

In [ ]:
Ms = 2**np.arange(2, 8)                  # 4, 8, 16, 32, 64, 128
hh, err = [], []

print('   M         h          error       razon')
for M in Ms:
    t0 = time.perf_counter()
    h, e = resolver(int(M))
    t = time.perf_counter() - t0
    razon = err[-1]/e if err else np.nan
    hh.append(h); err.append(e)
    print(f'{M:4d}  {h:9.6f}  {e:.4e}  {razon:8.3f}   ({t*1000:.0f} ms)')

hh, err = np.array(hh), np.array(err)

Las razones deben acercarse a **4**: eso es el orden 2.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(hh, err, 'r-o', label='err')
ax.loglog(hh, hh**2, 'k--', label='$h^2$')
ax.loglog(hh, hh, 'b--', label='$h$')
ax.set_xlabel('h')
ax.set_ylabel('error')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.show()

---

# Cómo NO armar los vectores

Acá está la diferencia importante entre MATLAB y Python, y vale la pena verla
medida.

En MATLAB, `ii = [ii, pos]` dentro de un ciclo es un problema serio: cada
asignación pide un bloque de memoria nuevo y copia todo lo anterior, de modo que
armar $N$ entradas cuesta $O(N^2)$. Por eso `MDF_v2_*.m` preasigna el espacio
exacto — y ahí la ganancia es real, del orden de 80×.

**En Python esa lección no se traslada.** Las listas ya reservan espacio de más,
así que `append` es $O(1)$ amortizado. Preasignar un arreglo de NumPy y llenarlo
con un contador casi no mejora nada.

Lo que sí es un desastre es traducir `ii = [ii, pos]` *literalmente* a
`np.append`, que copia el arreglo completo en cada llamada. Comparemos las tres
formas de armar exactamente el mismo triplete.

In [ ]:
def con_listas(M):
    """Listas de Python + append. Lo idiomatico."""
    ii, jj, ss = [], [], []
    for j in range(1, M):
        for i in range(1, M):
            pos = (j-1)*(M-1) + (i-1)
            ii.append(pos); jj.append(pos); ss.append(4.0)
            if i != 1:   ii.append(pos); jj.append(pos-1);     ss.append(-1.0)
            if i != M-1: ii.append(pos); jj.append(pos+1);     ss.append(-1.0)
            if j != 1:   ii.append(pos); jj.append(pos-(M-1)); ss.append(-1.0)
            if j != M-1: ii.append(pos); jj.append(pos+(M-1)); ss.append(-1.0)
    return np.array(ii), np.array(jj), np.array(ss)


def con_prealloc(M):
    """np.empty preasignado + contador. La traduccion de MDF_v2_*.m."""
    nnz = 5*(M-1)**2 - 4*(M-1)           # conteo exacto de entradas no nulas
    ii = np.empty(nnz, dtype=np.int64)
    jj = np.empty(nnz, dtype=np.int64)
    ss = np.empty(nnz)
    k = 0
    for j in range(1, M):
        for i in range(1, M):
            pos = (j-1)*(M-1) + (i-1)
            ii[k] = pos; jj[k] = pos; ss[k] = 4.0; k += 1
            if i != 1:   ii[k]=pos; jj[k]=pos-1;     ss[k]=-1.0; k+=1
            if i != M-1: ii[k]=pos; jj[k]=pos+1;     ss[k]=-1.0; k+=1
            if j != 1:   ii[k]=pos; jj[k]=pos-(M-1); ss[k]=-1.0; k+=1
            if j != M-1: ii[k]=pos; jj[k]=pos+(M-1); ss[k]=-1.0; k+=1
    assert k == nnz
    return ii, jj, ss


def con_npappend(M):
    """np.append dentro del ciclo: la traduccion LITERAL de ii = [ii, pos]."""
    ii = np.array([], dtype=np.int64)
    jj = np.array([], dtype=np.int64)
    ss = np.array([])
    for j in range(1, M):
        for i in range(1, M):
            pos = (j-1)*(M-1) + (i-1)
            ii = np.append(ii, pos); jj = np.append(jj, pos); ss = np.append(ss, 4.0)
            if i != 1:
                ii = np.append(ii, pos); jj = np.append(jj, pos-1);     ss = np.append(ss, -1.0)
            if i != M-1:
                ii = np.append(ii, pos); jj = np.append(jj, pos+1);     ss = np.append(ss, -1.0)
            if j != 1:
                ii = np.append(ii, pos); jj = np.append(jj, pos-(M-1)); ss = np.append(ss, -1.0)
            if j != M-1:
                ii = np.append(ii, pos); jj = np.append(jj, pos+(M-1)); ss = np.append(ss, -1.0)
    return ii, jj, ss

Primero, comprobar que las tres construyen **la misma matriz**. Si no, comparar
tiempos no tendría sentido.

In [ ]:
def armar(triplete, M):
    ii, jj, ss = triplete
    n = (M-1)**2
    return coo_matrix((ss, (ii, jj)), shape=(n, n)).tocsr()

Mc = 24
ref = armar(con_listas(Mc), Mc)
for nombre, fn in [('preasignado', con_prealloc), ('np.append', con_npappend)]:
    igual = (armar(fn(Mc), Mc) - ref).nnz == 0
    print(f'{nombre:12s} da la misma matriz: {igual}')

In [ ]:
metodos = [('listas + append', con_listas),
           ('np.empty preasignado', con_prealloc),
           ('np.append (literal)', con_npappend)]
Ms_t = [16, 32, 64]
tiempos = {n: [] for n, _ in metodos}

print(f'{"M":>4}  ' + '  '.join(f'{n:>22}' for n, _ in metodos))
for M in Ms_t:
    fila = []
    for nombre, fn in metodos:
        t0 = time.perf_counter(); fn(M); t = time.perf_counter() - t0
        tiempos[nombre].append(t*1000)
        fila.append(f'{t*1000:19.2f} ms')
    print(f'{M:4d}  ' + '  '.join(fila))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for nombre, _ in metodos:
    ax.loglog(Ms_t, tiempos[nombre], 'o-', label=nombre)
ax.set_xlabel('M')
ax.set_ylabel('tiempo de ensamblaje [ms]')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.show()

Las dos primeras curvas quedan casi encimadas y con la misma pendiente. La de
`np.append` va por arriba y **más inclinada**: esa pendiente mayor es el
$O(N^2)$, y significa que la distancia se agranda al refinar la malla.

La moraleja cambia de lenguaje:

| | qué importa |
|---|---|
| MATLAB | preasignar `ii`, `jj`, `ss` |
| Python | no salir del nivel de NumPy |

---

# La ruta corta

Una vez entendido el ensamblaje, conviene saber que en la práctica esta matriz
no se arma con ciclos. El laplaciano 2D es una suma de productos de Kronecker
del laplaciano 1D:

$$A = I \otimes T + T \otimes I, \qquad T = \mathrm{tridiag}(-1, 2, -1)$$

Tres líneas, sin ciclos, y el resultado es idéntico. Esconde justamente lo que
este notebook quiso mostrar, pero es lo que uno usa después.

In [ ]:
from scipy.sparse import diags, eye, kron

def laplaciano_kron(M):
    n = M - 1
    T = diags([-1, 2, -1], [-1, 0, 1], shape=(n, n))
    return (kron(eye(n), T) + kron(T, eye(n))).tocsr()

Mk = 24
print('identica a la ensamblada con ciclos:',
      (laplaciano_kron(Mk) - armar(con_listas(Mk), Mk)).nnz == 0)

t0 = time.perf_counter(); laplaciano_kron(128); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); con_listas(128);      t2 = time.perf_counter() - t0
print(f'kron      : {t1*1000:7.2f} ms')
print(f'con ciclos: {t2*1000:7.2f} ms')